# Master ML Lifecycle Notebook

Immutable reference notebook restored from the container image whenever the pod starts.

Copy useful cells into one of the scratch notebooks before experimenting.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Jupyter ML Workbench ready')
print('NumPy:', np.__version__)
print('pandas:', pd.__version__)


## 1. Create a sample structured dataset


In [ ]:
rng = np.random.default_rng(42)

df = pd.DataFrame({
    'age': rng.integers(18, 80, 1000),
    'income': rng.normal(80000, 25000, 1000).clip(15000),
    'transactions': rng.integers(0, 100, 1000),
    'credit_score': rng.integers(300, 850, 1000),
})

df['high_risk'] = (
    (df['credit_score'] < 600) |
    (df['income'] < 45000)
).astype(int)

df.head()


## 2. Explore and visualize


In [ ]:
display(df.describe())

df['credit_score'].hist(bins=30)
plt.title('Credit Score Distribution')
plt.xlabel('Credit score')
plt.ylabel('Count')
plt.show()


## 3. Train multiple ML models


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

features = ['age', 'income', 'transactions', 'credit_score']

X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df['high_risk'],
    test_size=0.2,
    random_state=42,
)

models = {
    'logistic': LogisticRegression(max_iter=1000),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'gradient_boosting': GradientBoostingClassifier(),
}


In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred),
        'recall': recall_score(y_test, pred),
        'f1': f1_score(y_test, pred),
    })

results_df = pd.DataFrame(results).sort_values('f1', ascending=False)
results_df


## 4. Plot model comparison


In [ ]:
results_df.plot(
    x='model',
    y=['accuracy', 'precision', 'recall', 'f1'],
    kind='bar',
)
plt.title('Model Comparison')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.show()


## 5. Select the winning model and create an artifact


In [ ]:
import joblib

best_name = results_df.iloc[0]['model']
best_model = models[best_name]

artifact_dir = Path('/home/jovyan/work/artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)

artifact_path = artifact_dir / 'risk_model.joblib'
joblib.dump(best_model, artifact_path)

print('Winning model:', best_name)
print('Artifact:', artifact_path)


## 6. Load the artifact again


In [ ]:
loaded_model = joblib.load('/home/jovyan/work/artifacts/risk_model.joblib')

sample = pd.DataFrame([{
    'age': 52,
    'income': 93000,
    'transactions': 31,
    'credit_score': 732,
}])

prediction = loaded_model.predict(sample)[0]
probability = loaded_model.predict_proba(sample)[0][1]

print('prediction:', int(prediction))
print('high-risk probability:', round(float(probability) * 100, 2), '%')


## 7. Example inference endpoint call


In [ ]:
import requests

endpoint = 'http://YOUR_INFERENCE_SERVICE/v1/predict'

payload = {
    'age': 52,
    'income': 93000,
    'transactions': 31,
    'credit_score': 732,
}

print('Set endpoint to a real Kubernetes Service URL before sending the request.')
print('endpoint =', endpoint)
print('payload =', payload)


## 8. Cloud artifact storage

The image already contains SDKs for AWS S3, Azure Blob Storage, and GCS.

Use workload identity or Vault-injected credentials rather than storing static cloud keys inside this notebook.


## 9. Additional experiments

Good scratch-notebook exercises:

- replace Random Forest with XGBoost
- add feature scaling
- add SHAP later
- build a small PyTorch network
- record metrics to MLflow
- call a FastAPI inference service
- wrap endpoint selection in LangGraph
